# 04. Cache / Persist

02_lazy_eval.ipynb에서 본 재계산 문제를 `.cache()`/`.persist()`로 해결하고, 두 번째 action이 빨라지는 것을 확인한다.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

spark = SparkSession.builder.appName("04_cache_persist").getOrCreate()

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("amount", DoubleType(), False),
])
orders = spark.read.csv("/opt/spark-data/orders.csv", header=True, schema=orders_schema)
chain = orders.filter(orders.amount > 100.0).select("user_id", "amount")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/27 01:19:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
chain.cache()

DataFrame[user_id: int, amount: double]

In [3]:
import time

start = time.time()
first_count = chain.count()
print(f"first count={first_count} elapsed={time.time() - start:.2f}s (캐시를 채우는 첫 실행)")

start = time.time()
second_count = chain.count()
print(f"second count={second_count} elapsed={time.time() - start:.2f}s (캐시에서 바로 읽음)")

first count=160431 elapsed=0.90s (캐시를 채우는 첫 실행)
second count=160431 elapsed=0.07s (캐시에서 바로 읽음)


Spark UI의 Storage 탭에서 이 DataFrame이 메모리에 캐시된 것을 확인할 수 있다. 두 번째 `count()`의 elapsed 시간이 눈에 띄게 줄어든다.

In [4]:
chain.unpersist()
spark.stop()